# BigQuery Read/Write Patterns on Databricks

This notebook covers the primary patterns for reading from and writing to Google BigQuery from a Databricks notebook. We use two libraries:

- **`google-cloud-bigquery`** (Python client) — metadata queries, DDL, small result sets, and anything that benefits from direct API control.
- **Spark BigQuery connector** (`spark.read.format("bigquery")`) — large-scale DataFrame reads and writes via the BigQuery Storage APIs.

Auth is handled by the compute profile. On a BigQuery-enabled cluster, ADC is wired before notebook code runs. **No credential code is needed here.**

## Configuration

In [ ]:
PROJECT = "idd-dscoe-nonprod-1"
DATASET = "p17851h"

FULL_TABLE = f"{PROJECT}.{DATASET}"

print(f"Project:  {PROJECT}")
print(f"Dataset:  {DATASET}")
print(f"Full path: {FULL_TABLE}")

## Setup

In [ ]:
from google.cloud import bigquery
from pyspark.sql import functions as F

bq_client = bigquery.Client(project=PROJECT)

---
## 1. Python Client — Metadata & Exploration

In [ ]:
tables = list(bq_client.list_tables(DATASET))
print(f"Tables in {DATASET}:")
for t in tables:
    print(f"  {t.table_id}  ({t.table_type})")

In [ ]:
table_ref = f"{FULL_TABLE}.{tables[0].table_id}" if tables else f"{FULL_TABLE}.example"
table = bq_client.get_table(table_ref)
print(f"Schema for {table_ref}:")
for field in table.schema:
    print(f"  {field.name:<30} {field.field_type:<10} {field.mode}")
print(f"\nRows: {table.num_rows:,}")

---
## 2. Python Client — Query to Pandas

Use the Python client when you want query results in a local pandas DataFrame. Best suited for small-to-medium result sets or when you need full control over query configuration.

In [ ]:
QUERY = f"""
SELECT table_name, table_type
FROM `{FULL_TABLE}.INFORMATION_SCHEMA.TABLES`
LIMIT 10
"""

df_pandas = bq_client.query(QUERY).to_dataframe()
display(df_pandas)

### Query with job config

Pass a `QueryJobConfig` to control labels, timeouts, destination tables, write disposition, and more.

In [ ]:
job_config = bigquery.QueryJobConfig(
    use_legacy_sql=False,
    labels={"team": "bi-onboarding", "source": "databricks"},
)

job = bq_client.query(QUERY, job_config=job_config)
print(f"Job ID: {job.job_id}")
print(f"Bytes processed: {job.total_bytes_processed:,}")
print(f"Slot millis: {job.slot_millis:,}")

---
## 3. Python Client — Write to BigQuery

Load a pandas DataFrame into BigQuery. This is ideal for smaller datasets or one-off loads.

In [ ]:
import pandas as pd

df_sample = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "carol"],
    "value": [10.5, 20.0, 30.5],
})

DESTINATION_TABLE = f"{DATASET}.onboarding_write_test"

job = bq_client.load_table_from_dataframe(
    df_sample,
    DESTINATION_TABLE,
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
)
job.result()
print(f"Wrote {len(df_sample)} rows to {DESTINATION_TABLE}")

In [ ]:
result = bq_client.query(f"SELECT * FROM `{FULL_TABLE}.onboarding_write_test`").to_dataframe()
display(result)

---
## 4. Python Client — DDL & Table Management

The Python client is the right tool for DDL operations that don't return data.

In [ ]:
bq_client.query(f"""
    CREATE TABLE IF NOT EXISTS `{FULL_TABLE}.onboarding_managed` (
        id INT64 NOT NULL,
        name STRING,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
    )
""").result()

print(f"Created {FULL_TABLE}.onboarding_managed")

In [ ]:
bq_client.query(f"DROP TABLE IF EXISTS `{FULL_TABLE}.onboarding_write_test`").result()
bq_client.query(f"DROP TABLE IF EXISTS `{FULL_TABLE}.onboarding_managed`").result()
print("Cleaned up test tables")

---
## 5. Spark Connector — Read a Table

The Spark BigQuery connector reads through the **BigQuery Storage Read API**, which is the high-throughput distributed path. Use this for large tables.

In [ ]:
source_table = f"{FULL_TABLE}.{tables[0].table_id}" if tables else f"{FULL_TABLE}.example"

df_spark = (
    spark.read.format("bigquery")
    .option("table", source_table)
    .load()
)

print(f"Rows: {df_spark.count():,}")
df_spark.printSchema()
display(df_spark.limit(5))

### Read with column selection and filter pushdown

The connector pushes down filters and column selection to BigQuery to minimize data transfer.

In [ ]:
cols = df_spark.columns[:3] if len(df_spark.columns) >= 3 else df_spark.columns

df_filtered = (
    spark.read.format("bigquery")
    .option("table", source_table)
    .option("filter", f"{cols[0]} IS NOT NULL")
    .load()
    .select(*cols)
)

display(df_filtered.limit(10))

### Read from a query

The connector can execute a SQL query and read the materialized result. Requires `viewsEnabled`. The connector materializes results into a temp table internally, so this is slightly more expensive than a direct table read.

In [ ]:
spark.conf.set("viewsEnabled", "true")

QUERY_SQL = f"SELECT * FROM `{source_table}` LIMIT 100"

df_query = (
    spark.read.format("bigquery")
    .option("query", QUERY_SQL)
    .load()
)

display(df_query.limit(10))

---
## 6. Spark Connector — Write to BigQuery

We use **`writeMethod=direct`** (BigQuery Storage Write API). This avoids the need for a GCS staging bucket and keeps the data path simple: `Spark -> Storage Write API -> BigQuery`.

In [ ]:
SPARK_DEST = f"{DATASET}.onboarding_spark_write"

df_to_write = spark.createDataFrame(
    [(1, "alice", 100.0), (2, "bob", 200.0), (3, "carol", 300.0)],
    "id INT, name STRING, value DOUBLE",
)

(
    df_to_write.write.format("bigquery")
    .option("writeMethod", "direct")
    .option("table", SPARK_DEST)
    .mode("overwrite")
    .save()
)

print(f"Wrote {df_to_write.count()} rows to {FULL_TABLE}.onboarding_spark_write")

In [ ]:
df_verify = (
    spark.read.format("bigquery")
    .option("table", f"{FULL_TABLE}.onboarding_spark_write")
    .load()
)
display(df_verify)

### Append mode

In [ ]:
df_more = spark.createDataFrame(
    [(4, "dave", 400.0), (5, "eve", 500.0)],
    "id INT, name STRING, value DOUBLE",
)

(
    df_more.write.format("bigquery")
    .option("writeMethod", "direct")
    .option("table", SPARK_DEST)
    .mode("append")
    .save()
)

total = spark.read.format("bigquery").option("table", f"{FULL_TABLE}.onboarding_spark_write").load()
print(f"Total rows after append: {total.count()}")
display(total)

---
## 7. Spark Connector — Transform and Write Back

A realistic pattern: read from BigQuery, transform with Spark, write the result to a new table.

In [ ]:
OUTPUT_TABLE = f"{DATASET}.onboarding_spark_transformed"

df_transformed = (
    spark.read.format("bigquery")
    .option("table", f"{FULL_TABLE}.onboarding_spark_write")
    .load()
    .withColumn("value_doubled", F.col("value") * 2)
    .withColumn("ingested_at", F.current_timestamp())
)

(
    df_transformed.write.format("bigquery")
    .option("writeMethod", "direct")
    .option("table", OUTPUT_TABLE)
    .mode("overwrite")
    .save()
)

display(
    spark.read.format("bigquery")
    .option("table", f"{FULL_TABLE}.onboarding_spark_transformed")
    .load()
)

---
## 8. Cleanup

In [ ]:
for tbl in ["onboarding_spark_write", "onboarding_spark_transformed"]:
    bq_client.query(f"DROP TABLE IF EXISTS `{FULL_TABLE}.{tbl}`").result()
    print(f"Dropped {tbl}")

---
## Quick Reference

| Task | Library | Pattern |
|---|---|---|
| List tables / inspect schema | Python client | `bq_client.list_tables()`, `bq_client.get_table()` |
| Run SQL, small results | Python client | `bq_client.query(sql).to_dataframe()` |
| DDL (CREATE, DROP, ALTER) | Python client | `bq_client.query(ddl).result()` |
| Load pandas to BigQuery | Python client | `bq_client.load_table_from_dataframe()` |
| Read large table into Spark | Spark connector | `spark.read.format("bigquery").option("table", ...).load()` |
| Read query result into Spark | Spark connector | `.option("query", sql)` with `viewsEnabled=true` |
| Write Spark DataFrame to BigQuery | Spark connector | `.write.format("bigquery").option("writeMethod", "direct")` |

**Key connector options:** `writeMethod=direct` (recommended), `filter`, `viewsEnabled`, `readDataFormat` (ARROW/AVRO).